In [2]:
# minimal test

import pylink

jlink = pylink.JLink()

for probe in jlink.connected_emulators():
    print("J-Link detected")
    print("Serial:", probe.SerialNumber)
    

J-Link detected
Serial: 600105462


In [3]:
import pylink

jlink = pylink.JLink()

probes = jlink.connected_emulators()

if not probes:
    print("FAIL: No J-Link found")
else:
    serial = probes[0].SerialNumber
    print(f"OK: J-Link found: {serial}")

    try:
        jlink.open(serial_no=serial)
        print("OK: J-Link USB communication works")

        jlink.set_tif(pylink.enums.JLinkInterfaces.SWD)
        jlink.connect("ATSAM4S2C", speed=1000)

        print("OK: SAM4S SWD connection works")
        print(f"Target voltage: {jlink.target_voltage():.3f} V")

    except Exception as e:
        print("FAIL:", repr(e))

    finally:
        if jlink.opened():
            jlink.close()


OK: J-Link found: 600105462
OK: J-Link USB communication works
OK: SAM4S SWD connection works
FAIL: AttributeError("'JLink' object has no attribute 'target_voltage'")


In [5]:
import pylink

SERIAL = 600105462
DEVICE = "ATSAM4S2C"

jlink = pylink.JLink()

try:
    print("Opening J-Link...")
    jlink.open(serial_no=SERIAL)

    print("S/N:", jlink.serial_number)

    print(f"\nConnecting to {DEVICE} using SWD...")

    jlink.set_tif(pylink.enums.JLinkInterfaces.SWD)
    jlink.connect(DEVICE)

    print("Connected:", jlink.connected())

    # -------------------------------------------------
    # Halt CPU before reading CPU registers
    # -------------------------------------------------

    print("\nHalting CPU...")
    jlink.halt()

    print("CPU halted:", jlink.halted())

    # Cortex-M registers
    r0 = jlink.register_read(0)
    sp = jlink.register_read(13)
    lr = jlink.register_read(14)
    pc = jlink.register_read(15)

    print()
    print(f"R0 : 0x{r0:08X}")
    print(f"SP : 0x{sp:08X}")
    print(f"LR : 0x{lr:08X}")
    print(f"PC : 0x{pc:08X}")

    # -------------------------------------------------
    # Read Cortex-M CPUID
    # -------------------------------------------------

    CPUID_ADDR = 0xE000ED00

    cpuid = jlink.memory_read32(CPUID_ADDR, 1)[0]

    print()
    print(f"CPUID: 0x{cpuid:08X}")

    print("\nJ-Link/SWD test PASSED")

    # -------------------------------------------------
    # Continue firmware execution
    # -------------------------------------------------

    print("\nRestarting CPU...")
    jlink.restart()

    print("CPU halted:", jlink.halted())

except Exception as e:
    print()
    print("J-Link test FAILED")
    print(type(e).__name__ + ":", e)

finally:
    jlink.close()
    print("\nJ-Link closed.")

Opening J-Link...
S/N: 600105462

Connecting to ATSAM4S2C using SWD...
Connected: True

Halting CPU...
CPU halted: True

R0 : 0x00000001
SP : 0x20001078
LR : 0x00400659
PC : 0x0040065A

CPUID: 0x410FC241

J-Link/SWD test PASSED

Restarting CPU...
CPU halted: False

J-Link closed.
